# CorrDiff Ensemble Inference Workflow - Modular Version

This notebook demonstrates an advanced ensemble workflow for CorrDiff using modular Python scripts.
All core functionality has been extracted into reusable modules in the `src/` directory.

## Features
- **Modular Design**: Core functionality organized in separate modules
- **Easy Configuration**: Centralized configuration management
- **Ensemble Generation**: Multiple realizations with different seeds
- **Comprehensive Analysis**: Statistical analysis and visualization
- **NetCDF Output**: PhysicsNeMo-compatible output format

## Workflow Overview
1. Load configuration and setup
2. Initialize data source and models
3. Run ensemble inference
4. Save results and analyze statistics
5. Create visualizations

In [1]:
! chmod -R 777 /app/outputs/generation/

## Import

In [2]:
import os
import sys
import torch

# Add source modules to path
src_path = '/app/host/home/younes.abid/git/earth2studio/notebooks/tutorials/cordiff/src'
sys.path.append(src_path)

# Import modular components
from trim_coordinates import trim_coordinates
from config import EnsembleConfig
from data_loader import create_data_source
from ensemble_model import create_ensemble_model
from inference import run_ensemble_inference
from output import save_ensemble_netcdf, verify_output
from stats import compute_and_save_stats
from visualization import plot_ensemble_analysis, print_ensemble_summary

print('✅ All modular components imported successfully')

✅ All modular components imported successfully


## Trimm coord for reference (should be done only one time)

In [3]:
# # Trim WRF coordinates to 432x432 for CorrDiff
# wrf_path = "/app/host/home/younes.abid/git/physicsnemo/data/georefrenced/Space42_CorrDiff/wrf_coord.nc"
# trim_pixels = (7, 8, 7, 8)  # (top, bottom, left, right)
# output_path = "/app/host/home/younes.abid/git/physicsnemo/data/georefrenced/Space42_CorrDiff/trimmed_coordinates_432x432.nc"

# trim_coordinates(wrf_path, trim_pixels, output_path)

## Setup and Configuration

In [4]:
# Create configuration object
config = EnsembleConfig(variables="Fog_index")
config.NUMBER_OF_STEPS = 20

# Print configuration summary
config.print_config()

✅ Ensemble configuration loaded
📊 Ensemble setup: 4 members, stochastic sampling
🔄 Diffusion steps: 20, solver: euler
💾 Ensemble output: /app/outputs/generation/Fog_index/20260205_070006/ensemble_Fog_index_20260205_070006.nc
📁 Analysis folder: /app/outputs/generation/Fog_index/20260205_070006/analysis


## Data Source and Model Setup

In [5]:
# Create data source
data_source = create_data_source(config)

print('✅ Data source created')
print(f'📊 Available samples: {data_source.total_samples}')
print(f'📐 Input grid shape: {data_source.input_shape}')

✅ Data source created
📊 Available samples: 504
📐 Input grid shape: (432, 432)


In [6]:
# Create ensemble CorrDiff model
ensemble_model = create_ensemble_model(config, data_source)

/usr/local/lib/python3.11/dist-packages/physicsnemo/models/module.py:460: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_dict = torch.load(


✅ Regression model loaded in 1.36s
   📊 Parameters: 78,353,281 total, 78,353,281 trainable
   💾 Memory usage: 7120.2 MB
✅ Diffusion model loaded in 1.50s
   📊 Parameters: 78,354,433 total, 78,354,433 trainable
   💾 Memory usage: 7231.3 MB


## Ensemble Inference

In [7]:
# Run ensemble inference
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
results = run_ensemble_inference(config, ensemble_model, data_source, device)

print(f'📊 Inference complete: {len(results["predictions"])} time steps with {config.NUM_ENSEMBLES} ensemble members each')

🚀 Running inference: 2 time steps, 4 ensemble members on cuda device


/usr/local/lib/python3.11/dist-packages/physicsnemo/models/diffusion/layers.py:701: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(enabled=self.amp_mode):


   Progress: 1/2 steps completed


/usr/local/lib/python3.11/dist-packages/physicsnemo/models/diffusion/layers.py:701: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(enabled=self.amp_mode):


   Progress: 2/2 steps completed
✅ Inference complete: 2/2 steps successful
📊 Inference complete: 2 time steps with 4 ensemble members each


## Save Results and Analysis

In [8]:
# Save results in NetCDF format
saved_file = save_ensemble_netcdf(config, results, ensemble_model)

# Verify the saved file
verify_output(saved_file)

💾 Saving to: ensemble_Fog_index_20260205_070006.nc
✅ Saved: 2 times, 4 members
📁 Verifying: ensemble_Fog_index_20260205_070006.nc
   ✅ prediction: {'y': 432, 'x': 432, 'ensemble': 4, 'time': 2} - ['lat', 'lon', 'Fog_index']
   ✅ input: {'y': 36, 'x': 40, 'time': 2} - ['lat', 'lon', 't_850', 't_500', 'z_850', 'z_500', 'u_850', 'u_500', 'v_850', 'v_500', 'u10', 'v10', 't2m', 'd2m', 'skt', 'sp', 'tcwv', 'tp']
   ✅ truth: {'y': 432, 'x': 432, 'time': 2} - ['lat', 'lon', 'Fog_index']


/app/host/home/younes.abid/git/earth2studio/notebooks/tutorials/cordiff/src/output.py:212: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f'   ✅ {group_name}: {dict(ds.dims)} - {list(ds.data_vars.keys())}')
/app/host/home/younes.abid/git/earth2studio/notebooks/tutorials/cordiff/src/output.py:212: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f'   ✅ {group_name}: {dict(ds.dims)} - {list(ds.data_vars.keys())}')
/app/host/home/younes.abid/git/earth2studio/notebooks/tutorials/cordiff/src/output.py:212: FutureWarning: The return type of `Dataset.dims` will be changed to return a set 

In [ ]:
# Compute and save ensemble statistics
summary_stats, stats_file = compute_and_save_stats(config, saved_file)

In [ ]:
summary_stats

## Visualization and Analysis

In [ ]:
# Create comprehensive ensemble visualizations
plot_ensemble_analysis(config, saved_file, time_idx=1, show=True)

In [ ]:
# Load and inspect the saved data
import xarray as xr

# Open prediction dataset
pred_ds = xr.open_dataset(saved_file, group='prediction')
pred_ds

## Summary and Next Steps

In [ ]:
# Print comprehensive summary
print_ensemble_summary(config, results, saved_file, summary_stats)

## Advanced Configuration Options

You can easily modify the ensemble configuration by creating a new config object:

```python
# Example: Different sampling configuration
custom_config = EnsembleConfig(variables="Fog_index")
custom_config.SAMPLING_MODE = 'deterministic'
custom_config.NUM_ENSEMBLES = 16
custom_config.NUMBER_OF_STEPS = 50
custom_config.INFERENCE_TIMES = ['2024-05-01T00:00:00', '2024-05-02T00:00:00']

# Run with custom configuration
# data_source = create_data_source(custom_config)
# ensemble_model = create_ensemble_model(custom_config, data_source)
# results = run_ensemble_inference(custom_config, ensemble_model, data_source)
```

## Module Structure

The workflow is organized into these modules in `src/`:

- **`config.py`**: Configuration management and parameter settings
- **`data_loader.py`**: Custom data source for PhysicsNeMo format
- **`ensemble_model.py`**: Ensemble CorrDiff model implementation
- **`inference.py`**: Inference workflow and execution
- **`output.py`**: NetCDF output and statistical analysis
- **`visualization.py`**: Plotting and visualization functions

This modular design makes it easy to:
- Modify individual components without affecting others
- Reuse functions across different notebooks
- Test and debug specific functionality
- Extend the workflow with new features

## Troubleshooting

If you encounter visualization issues with cartopy (geographic projections), the visualization module automatically uses simple matplotlib plots instead. This provides the same ensemble analysis without geographic context but is more reliable across different environments.

The modular design allows you to easily swap out components or add new functionality as needed.

In [ ]:
# Let's trace what map_coords actually does in your case
import numpy as np
from collections import OrderedDict

# Simulate your data loader coordinates (from CustomCorrDiffDataSource)
data_source_coords = OrderedDict({
    'time': np.array(['2024-04-30T00:00:00'], dtype='datetime64[ns]'),
    'variable': np.array(['t_850', 't_500', 'z_850', 'z_500']),  # 16 variables
    'lat': np.linspace(19.25, 28.0, 36),  # Input resolution
    'lon': np.linspace(116.0, 126.0, 40)   # Input resolution
})

# Simulate your ensemble model coordinates
ensemble_model_coords = OrderedDict({
    'batch': np.empty(0),
    'variable': np.array(['t_850', 't_500', 'z_850', 'z_500']),  # Same variables
    'lat': np.linspace(19.25, 28.0, 36),  # Same lat
    'lon': np.linspace(116.0, 126.0, 40),  # Same lon
})

print('=== COORDINATE COMPARISON ===')
print('Data source lat shape:', data_source_coords['lat'].shape)
print('Data source lat values:', data_source_coords['lat'][:3], '...', data_source_coords['lat'][-3:])
print()
print('Ensemble model lat shape:', ensemble_model_coords['lat'].shape) 
print('Ensemble model lat values:', ensemble_model_coords['lat'][:3], '...', ensemble_model_coords['lat'][-3:])
print()
print('Are lat coordinates identical?', np.array_equal(data_source_coords['lat'], ensemble_model_coords['lat']))
print('Are lon coordinates identical?', np.array_equal(data_source_coords['lon'], ensemble_model_coords['lon']))
print('Are variables identical?', np.array_equal(data_source_coords['variable'], ensemble_model_coords['variable']))
